1. **Konteks Data**

Dataset berikut ini berasal dari survei mengenai "Kesehatan Mental di Industri Teknologi". Fokus utama yang ada pada data ini adalah untuk memahami bagaimana pandangan karyawan terhadap kesehatan mental dan sejauh mana perusahaan menyediakan dukungan.
Tujuan digunakannya data ini adalah untuk memprediksi apakah seorang karyawan akan mencari bantuan medis (treatment) berdasarkan latar belakang dan lingkungan kerja mereka.



Data mencakup informasi demografi (Usia, Gender), kondisi kerja (ukuran perusahaan, kerja remote), dan fasilitas kesehatan mental yang disediakan oleh pemberi kerja.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Memuat dataset
df = pd.read_csv('/content/survey.csv')

# Menampilkan informasi dataset dan 5 baris pertama
print("Struktur Data Mentah:")
df.info()
display(df.head())

Struktur Data Mentah:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1259 entries, 0 to 1258
Data columns (total 27 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Timestamp                  1259 non-null   object
 1   Age                        1259 non-null   int64 
 2   Gender                     1259 non-null   object
 3   Country                    1259 non-null   object
 4   state                      744 non-null    object
 5   self_employed              1241 non-null   object
 6   family_history             1259 non-null   object
 7   treatment                  1259 non-null   object
 8   work_interfere             995 non-null    object
 9   no_employees               1259 non-null   object
 10  remote_work                1259 non-null   object
 11  tech_company               1259 non-null   object
 12  benefits                   1259 non-null   object
 13  care_options               1259 non-null 

,Timestamp,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence,comments
0,2014-08-27 11:29:31,37,Female,United States,IL,NaN,No,Yes,Often,6-25,No,Yes,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,No,NaN
1,2014-08-27 11:29:37,44,M,United States,IN,NaN,No,No,Rarely,More than 1000,No,No,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,No,No,Don't know,No,NaN
2,2014-08-27 11:29:44,32,Male,Canada,NaN,NaN,No,No,Rarely,6-25,No,Yes,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,No,NaN
3,2014-08-27 11:29:46,31,Male,United Kingdom,NaN,NaN,Yes,Yes,Often,26-100,No,Yes,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,Yes,NaN
4,2014-08-27 11:30:22,31,Male,United States,TX,NaN,No,No,Never,100-500,Yes,Yes,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,No,NaN


Dataset terdiri dari 1.259 baris dan 27 kolom, yang sebagian besar data pada kolom didominasi oleh teks dan hanya 1 kolom numerik pada tabel usia. Sehingga memerlukan proses encoding  untuk mengubah data teks menjadi angka.


Terdapat *Missing Value* pada kolom *comments* sebanyak lebih dari 80% sehingga dilakukan pembersihan data yang kosong tersebut. Sementara itu, pada kolom *state* dan *work_interfere* perlu dilakukan imputasi data.

2. **Pembersihan Data**

In [ ]:
# Menghapus kolom yang tidak digunakan untuk analisis

df.drop(['Timestamp', 'comments', 'state'], axis=1, inplace=True)

# Menghapus data umur (Age) yang tidak masuk akal (Outlier)

df = df[(df['Age'] > 18) & (df['Age'] < 100)]

print("Data setelah kolom dibuang dan umur difilter:")
display(df.describe())

Data setelah kolom dibuang dan umur difilter:


,Age
count,1244.000000
mean,32.155949
std,7.231587
min,19.000000
25%,27.000000
50%,31.000000
75%,36.000000
max,72.000000


Pada tahap ini, dilakukan pembersihan kolom yang tidak diunakan pada analisis dan menangani outlier. Tujuannya untuk menghasilkan data yang bersih dan relevan terhadap kesehatan mental.

Terdapat 3 kolom yang dihapus yaitu


1.   Timestamp, karena tidak mempengaruhi terhadap kondisi mental
2.   Comments, terlalu banyak data yang kosong sekitar lebih dari 80%
3. State, responden yang dicari hanya di US dan selain itu dianggap sebagai nilai kosong





3. **Standarisasi Gender**

In [25]:
# Mengubah semua teks menjadi huruf kecil dan menghapus spasi di awal/akhir
df['Gender'] = df['Gender'].str.lower().str.strip()

# Mendefinisikan daftar variasi untuk setiap kategori
male_list = ['male', 'm', 'male-ish', 'maile', 'mal', 'male (cis)', 'make', 'male ', 'man', 'msle', 'mail', 'malr', 'cis man', 'cis male']
female_list = ['female', 'f', 'woman', 'femake', 'female ', 'cis-female/femme', 'female (cis)', 'femail', 'cis female']

# Fungsi pengelompokan
df['Gender'] = df['Gender'].apply(lambda x: 'male' if x in male_list else ('female' if x in female_list else 'other'))

# Menampilkan hasil unik untuk verifikasi
print("Kategori Gender unik setelah proses normalisasi:")
print(df['Gender'].unique())

# Menampilkan distribusi jumlah data per gender
print("\nDistribusi Gender:")
print(df['Gender'].value_counts())

Kategori Gender unik setelah proses normalisasi:
['female' 'male' 'other']

Distribusi Gender:
Gender
male      981
female    246
other      17
Name: count, dtype: int64


Untuk output di atas, menunjukkan bahwa kolom Gender hanya memiliki 3 kode yaitu "Female", "Male", dan "other". Dengan melakukan tahap ini, kita dapat memberikan model yang efisien dan lebih kuat karena jumlah data atau sampel sudah terkumpul sesuai kategori yang ditentukan.

Melalui fungsi value_counts(), kita bisa mengetahui distribusi seluruh responden, dan informasi ini memberikan gambaran awal apakah data ini seimbang atau lebih didominasi oleh gender tertentu, yang nantinya dapat mempengaruhi pada saat pengambilan keputusan.

4. **Imputasi Nilai Kosong**

Menurut buku Sarkar dkk., dalam alogritma machine learning itu, tidak boleh membiarkan dataset memiliki nilai kosong. Oleh karena itu, pada kolom "self_employed" di isi dengan (no) karena mayoritas responden bukan self employed). Untuk kolom "work_interfere" di isi dengan (unknown). Hal ini dilakukan untuk mempertahankan data tanpa harus menebak kondisi mental tiap respondennya.

In [ ]:
# Mengisi nilai kosong pada self_employed dengan 'No' (modus)
df['self_employed'] = df['self_employed'].fillna('No')

# Mengisi nilai kosong pada work_interfere dengan 'Unknown'
df['work_interfere'] = df['work_interfere'].fillna('Unknown')

print("Cek jumlah nilai kosong (harus 0):")
print(df.isnull().sum().sum())

Cek jumlah nilai kosong (harus 0):
0


Output menunjukkan bahwa semua kolom sekarang memiliki angka 0 pada jumlah *Missing Value*. Dengan memberikan label "unknown" maka kita sudah memberikan informasi bahwa terdapat kelompok responden yang memilih untuk tidak menjawab.

5. **Transformasi Data(Encoding)**

Menurut Sarkar dkk,. mengatakan bahwa komputer hanya bisa memproses angka secara matematis, sehingga data yang masih berup teks harus diubah menjadi angka



In [26]:
# 1. Label Encoding untuk variabel target 'treatment' (Yes = 1, No = 0)
le = LabelEncoder()
df['treatment'] = le.fit_transform(df['treatment'])

# 2. One-Hot Encoding untuk semua kolom kategorikal lainnya
# Ini akan memecah kategori menjadi kolom-kolom biner (0 dan 1)
df_final = pd.get_dummies(df)

print("Informasi Dataset Akhir:")
print(f"Jumlah baris: {df_final.shape[0]}")
print(f"Jumlah kolom: {df_final.shape[1]}")
display(df_final.head())

Informasi Dataset Akhir:
Jumlah baris: 1244
Jumlah kolom: 113


,Age,treatment,Gender_female,Gender_male,Gender_other,Country_Australia,Country_Austria,Country_Belgium,Country_Bosnia and Herzegovina,Country_Brazil,Country_Bulgaria,Country_Canada,Country_China,Country_Colombia,Country_Costa Rica,Country_Croatia,Country_Czech Republic,Country_Denmark,Country_Finland,Country_France,Country_Georgia,Country_Germany,Country_Greece,Country_Hungary,Country_India,Country_Ireland,Country_Israel,Country_Italy,Country_Japan,Country_Latvia,Country_Mexico,Country_Moldova,Country_Netherlands,Country_New Zealand,Country_Nigeria,Country_Norway,Country_Philippines,Country_Poland,Country_Portugal,Country_Romania,...,care_options_No,care_options_Not sure,care_options_Yes,wellness_program_Don't know,wellness_program_No,wellness_program_Yes,seek_help_Don't know,seek_help_No,seek_help_Yes,anonymity_Don't know,anonymity_No,anonymity_Yes,leave_Don't know,leave_Somewhat difficult,leave_Somewhat easy,leave_Very difficult,leave_Very easy,mental_health_consequence_Maybe,mental_health_consequence_No,mental_health_consequence_Yes,phys_health_consequence_Maybe,phys_health_consequence_No,phys_health_consequence_Yes,coworkers_No,coworkers_Some of them,coworkers_Yes,supervisor_No,supervisor_Some of them,supervisor_Yes,mental_health_interview_Maybe,mental_health_interview_No,mental_health_interview_Yes,phys_health_interview_Maybe,phys_health_interview_No,phys_health_interview_Yes,mental_vs_physical_Don't know,mental_vs_physical_No,mental_vs_physical_Yes,obs_consequence_No,obs_consequence_Yes
0,37,1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,True,False,True,False,False,False,False,True,True,False
1,44,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,True,False,False,False,True,False,True,False,False,True,False,False,False,True,False,False,True,False,True,False,False,True,False
2,32,0,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,True,False,False,False,True,False,False,True,False,True,False,False,False,True,False,False,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,True,False,True,False,True,False
3,31,1,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,False,False,True,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,True,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True
4,31,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,...,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,True,False,False,True,False


Output dari kolom "treatment" berubah menjadi angka 1 (Yes) dan 0 (No), memudahkan model dalam melakukan klasifikasi biner. Dengan melalui *One-Hot Encoding* jumlah kolom menjadi bertambah karena setiap negara atau ukuran perusahaan memiliki kolom masing-masing.

Dengan format ini, datset sudah memenuhi beberapa tahapan untuk dimasukkan pada algoritma machine learning.